# ESG Filter (Tier A keywords + Tier B OpenRouter LLM)

Notebook này lọc các bài báo theo chủ đề **ESG** theo 2 tầng:
- **Tier A**: lọc ứng viên bằng từ điển ESG (từ file `ESG.xlsx`)
- **Tier B**: dùng **LLM OpenAI qua OpenRouter** để xác nhận bài có nói về **E hoặc S hoặc G** và trích `context_esg` + `evidence_quotes`

**Input/Output (silver 1/2/3):**
- Input mặc định: `data/<YEAR>/<BANK>/silver/2/*.json` (sau bước dedupe)
- Output: `data/<YEAR>/<BANK>/silver/3/esg_silver3_*.json`

Lưu ý:
- điền `OPENROUTER_API_KEY`.
- Nếu input thiếu `content`, notebook sẽ cố gắng crawl bổ sung (có thể bị chặn/429).

In [1]:
# Nếu thiếu package, chạy cell này.
%pip -q install pandas openpyxl requests trafilatura newspaper3k python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ===== CẤU HÌNH NĂM VÀ NGÂN HÀNG CẦN XỬ LÝ =====
# TODO: Thay đổi năm dữ liệu và tên ngân hàng bạn muốn xử lý ở đây
TARGET_YEAR = "2024"
TARGET_BANK = "LP Bank"  # Đổi thành "vietcombank", "bidv", v.v. hoặc "*" nếu muốn chạy tất cả ngân hàng trong năm
# ===============================================

In [3]:
from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Iterable
from urllib.parse import urlparse

import pandas as pd
import requests
import trafilatura
from newspaper import Article


In [4]:
# (Optional) Load API key from .env if you use one.
# .env example: OPENROUTER_API_KEY=xxxx
try:
    from dotenv import load_dotenv
    load_dotenv(override=True)
    print("Loaded .env (if present).")
except ModuleNotFoundError:
    print("python-dotenv not installed. Run the pip install cell above to install it.")

# Show whether key is present (masked)
_k = (os.getenv("OPENROUTER_API_KEY") or "").strip()
print("OPENROUTER_API_KEY present:", bool(_k))
if _k:
    print("OPENROUTER_API_KEY (masked):", _k[:6] + "..." + _k[-4:])

Loaded .env (if present).
OPENROUTER_API_KEY present: False


## 1) Cấu hình

In [5]:

# ===== OpenRouter / OpenAI (qua OpenRouter) =====
# Cách 1: điền trực tiếp vào biến OPENROUTER_API_KEY bên dưới (để trống nếu bạn muốn tự set sau)
# Cách 2: dùng file .env với dòng: OPENROUTER_API_KEY=xxxx (Cell trước sẽ load .env)
OPENROUTER_API_KEY = os.getenv("OPENAI_API_KEY")  # để trống; ưu tiên set qua .env/env var OPENROUTER_API_KEY
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "openai/gpt-4o-mini"  

# Rate limit cho LLM
LLM_SLEEP_SECONDS = 1.0
LLM_TIMEOUT_SECONDS = 60
LLM_MAX_RETRIES = 3

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() in {"thread_1", "thread_2"}:
    pass

# Trỏ thẳng đến thư mục data của năm được chỉ định
DATA_ROOT = PROJECT_ROOT / "data" / TARGET_YEAR

# Silver layout: silver/1 (raw crawl) -> silver/2 (after dedupe) -> silver/3 (after ESG filter)
# STAGE_IN = "silver/2"
STAGE_IN = "silver/2"
STAGE_OUT = "silver/3"

# Nếu STAGE_IN = bronze: chỉ xử lý những file Excel có đuôi này trong bronze
FILTER_EXCEL_SUFFIX = "_filter.xlsx"

# ESG dictionary (Tier A) - trỏ ra ngoài gốc NCKH vì thư mục ESG_Dictionary nằm ở ngoài
ESG_DICT_XLSX = PROJECT_ROOT.parent / "ESG_Dictionary" / "ESG.xlsx"

# ESG definition text (để nhúng vào prompt Tier B)
ESG_DEFINITION_TXT = PROJECT_ROOT / "Definition_ESG.txt"

# Crawl (chỉ dùng khi input không có content sẵn)
HTTP_TIMEOUT_SECONDS = 25
CRAWL_SLEEP_SECONDS = 0.8

# Cắt ngắn content gửi lên LLM để tiết kiệm token
MAX_CONTENT_CHARS_FOR_LLM = None

# (Optional) Giới hạn số bài để test nhanh; None = không giới hạn
MAX_ARTICLES_PER_BANK = None

## 2) Tier A — Load từ điển ESG + hàm lọc ứng viên

In [6]:
def load_esg_keywords(xlsx_path: Path) -> list[str]:
    if not xlsx_path.exists():
        raise FileNotFoundError(f"ESG dictionary not found: {xlsx_path}")
    df = pd.read_excel(xlsx_path, engine="openpyxl")
    if df.empty:
        raise ValueError(f"ESG dictionary is empty: {xlsx_path}")

    first_col = df.columns[0]
    raw = df[first_col].dropna().astype(str).tolist()
    # Bỏ header (dòng đầu tiên) nếu bị lẫn vào data
    raw = [s.strip() for s in raw if str(s).strip()]
    # Heuristic: bỏ item quá ngắn (<=1 ký tự) để giảm noise
    keywords = [s for s in raw if len(s) > 1]

    # Dedupe (casefold)
    seen = set()
    out = []
    for kw in keywords:
        key = kw.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(kw)

    return out


ESG_KEYWORDS = load_esg_keywords(ESG_DICT_XLSX)
print("Loaded ESG keywords:", len(ESG_KEYWORDS))
print("Sample:", ESG_KEYWORDS[:20])

Loaded ESG keywords: 798
Sample: ['thân thiện với môi trường', 'xanh', 'tái tạo', 'không thể tái tạo', 'ít tác động', 'tác động thấp', 'có trách nhiệm', 'bền vững', 'đạo đức', 'phi đạo đức', 'không có đạo đức', 'trách nhiệm', 'công bằng', 'chính trực', 'bình đẳng', 'đúng đắn', 'sinh học', 'hữu cơ', 'hoàn toàn tự nhiên', 'tự nhiên']


In [7]:
def build_keyword_regex_chunks(keywords: list[str], chunk_size: int = 80) -> list[re.Pattern]:
    # Dùng regex chunk để tránh pattern quá dài.
    pats: list[re.Pattern] = []
    escaped = [re.escape(k.strip()) for k in keywords if k.strip()]
    for i in range(0, len(escaped), chunk_size):
        chunk = escaped[i : i + chunk_size]
        if not chunk:
            continue
        pat = re.compile(r"(?:" + "|".join(chunk) + r")", flags=re.IGNORECASE)
        pats.append(pat)
    return pats


KEYWORD_PATTERNS = build_keyword_regex_chunks(ESG_KEYWORDS)
print("Keyword regex chunks:", len(KEYWORD_PATTERNS))


Keyword regex chunks: 10


In [8]:
def tier_a_find_hits(text: str, max_hits: int = 30) -> list[str]:
    if not text:
        return []
    hits: list[str] = []
    for pat in KEYWORD_PATTERNS:
        for m in pat.finditer(text):
            kw = m.group(0)
            if kw:
                hits.append(kw)
                if len(hits) >= max_hits:
                    return hits
    return hits


def extract_context_sentences(text: str, hits: list[str], max_sentences: int = 8) -> str:
    # Tách câu đơn giản theo dấu câu. Mục tiêu: lấy đoạn liên quan ESG để gửi lên LLM.
    if not text:
        return ""
    # Chuẩn hoá xuống dòng
    t = re.sub(r"\s+", " " , text).strip()
    if not t:
        return ""

    # Split câu (heuristic)
    sentences = re.split(r"(?<=[\.\!\?])\s+", t)
    if not hits:
        return " ".join(sentences[:max_sentences])

    hits_cf = [h.casefold() for h in hits]
    picked: list[str] = []
    for s in sentences:
        s_cf = s.casefold()
        if any(h in s_cf for h in hits_cf):
            picked.append(s)
            if len(picked) >= max_sentences:
                break

    if not picked:
        picked = sentences[:max_sentences]

    return " ".join(picked).strip()


## 3) Crawl nội dung bài báo (trafilatura, fallback newspaper3k)

In [9]:
def _strip_trailing_punct(url: str) -> str:
    return url.strip().rstrip(")].,;\"' ")


def fetch_article_text(url: str, session: requests.Session, timeout: int) -> tuple[str | None, str | None]:
    """Return (content, method). method in {trafilatura,newspaper3k} or None."""
    url = _strip_trailing_punct(url)

    # 1) trafilatura
    downloaded = None
    try:
        downloaded = trafilatura.fetch_url(url)
    except Exception:
        downloaded = None

    if not downloaded:
        try:
            resp = session.get(url, timeout=timeout, allow_redirects=True)
            resp.raise_for_status()
            downloaded = resp.text
        except Exception:
            downloaded = None

    if downloaded:
        try:
            text = trafilatura.extract(
                downloaded,
                url=url,
                include_comments=False,
                include_tables=False,
                favor_precision=True,
            )
            if text and len(text.strip()) >= 200:
                return text.strip(), "trafilatura"
        except Exception:
            pass

    # 2) newspaper3k fallback
    try:
        art = Article(url, language="vi")
        art.download()
        art.parse()
        text = (art.text or "").strip()
        if len(text) >= 200:
            return text, "newspaper3k"
    except Exception:
        pass

    return None, None

## 4) Load input (mặc định từ `silver/2/`; fallback `bronze/` nếu bạn đổi `STAGE_IN`)

In [10]:
@dataclass
class InputArticle:
    year: str
    bank: str
    link: str
    title: str
    source: str
    content: str | None = None


def _infer_bank_dir_from_path(p: Path) -> Path:
    # - JSON: .../data/<YEAR>/<BANK>/silver/2/<file>.json -> parents[2]
    # - XLSX: .../data/<YEAR>/<BANK>/bronze/<file>.xlsx -> parent.parent
    parts = [x.lower() for x in p.parts]
    if "silver" in parts:
        i = parts.index("silver")
        # .../<BANK>/silver/...
        if i >= 1:
            return Path(*p.parts[:i])
    return p.parent.parent


def iter_stage_input_paths(data_root: Path, target_bank: str = "*") -> Iterable[Path]:
    bank_path = "**" if target_bank == "*" else target_bank
    if str(STAGE_IN).casefold().startswith("silver"):
        pattern = f"{bank_path}/{STAGE_IN}/*.json"
    else:
        pattern = f"{bank_path}/{STAGE_IN}/*{FILTER_EXCEL_SUFFIX}"
    for p in data_root.glob(pattern):
        if not p.is_file():
            continue
        if p.name.startswith("~$"):
            continue
        yield p


def load_articles_from_bronze_excel(xlsx_path: Path) -> list[InputArticle]:
    # xlsx_path: .../data/<YEAR>/<BANK>/bronze/*.xlsx
    bank_dir = xlsx_path.parent.parent
    bank = bank_dir.name
    year = bank_dir.parent.name

    df = pd.read_excel(xlsx_path, engine="openpyxl")
    if df.empty:
        return []

    url_col = "url_out" if "url_out" in df.columns else ("url" if "url" in df.columns else None)
    if url_col is None:
        raise KeyError(f"Missing url_out/url in: {xlsx_path}")
    title_col = "title" if "title" in df.columns else None
    if title_col is None:
        raise KeyError(f"Missing title in: {xlsx_path}")

    out: list[InputArticle] = []
    for _, row in df.iterrows():
        link = str(row.get(url_col) or "").strip()
        title = str(row.get(title_col) or "").strip()
        if not link or link.lower() in {"nan", "none"}:
            continue
        if not title or title.lower() in {"nan", "none"}:
            continue
        link = _strip_trailing_punct(link)
        src = (urlparse(link).netloc or "").lower()
        out.append(InputArticle(year=year, bank=bank, link=link, title=title, source=src, content=None))

    # dedupe by link
    seen = set()
    dedup: list[InputArticle] = []
    for a in out:
        if a.link in seen:
            continue
        seen.add(a.link)
        dedup.append(a)

    if MAX_ARTICLES_PER_BANK is not None:
        dedup = dedup[: int(MAX_ARTICLES_PER_BANK)]
    return dedup


def load_articles_from_silver_json(json_path: Path) -> list[InputArticle]:
    # json_path: .../data/<YEAR>/<BANK>/silver/2/*.json (or silver/1, silver/3)
    bank_dir = json_path.parents[2]  # .../<YEAR>/<BANK>
    bank = bank_dir.name
    year = bank_dir.parent.name

    obj = json.loads(json_path.read_text(encoding="utf-8"))
    if not isinstance(obj, dict):
        raise ValueError(f"JSON root must be dict {{bank: [items]}}: {json_path}")

    out: list[InputArticle] = []
    for _bank_key, items in obj.items():
        if not isinstance(items, list):
            continue
        for it in items:
            if not isinstance(it, dict):
                continue
            link = str(it.get("link") or "").strip()
            title = str(it.get("title") or "").strip()
            content = it.get("content")
            content = str(content) if content is not None else ""
            src = str(it.get("source") or "").strip()
            if not link or link.lower() in {"nan", "none"}:
                continue
            if not title or title.lower() in {"nan", "none"}:
                continue
            link = _strip_trailing_punct(link)
            if not src:
                src = (urlparse(link).netloc or "").lower()
            out.append(InputArticle(year=year, bank=bank, link=link, title=title, source=src, content=content))

    # dedupe by link
    seen = set()
    dedup: list[InputArticle] = []
    for a in out:
        if a.link in seen:
            continue
        seen.add(a.link)
        dedup.append(a)

    if MAX_ARTICLES_PER_BANK is not None:
        dedup = dedup[: int(MAX_ARTICLES_PER_BANK)]
    return dedup


input_paths = sorted(iter_stage_input_paths(DATA_ROOT, TARGET_BANK))
print(f"Found input paths for STAGE_IN='{STAGE_IN}' (Bank='{TARGET_BANK}'):", len(input_paths))
for p in input_paths[:10]:
    print("-", p)

Found input paths for STAGE_IN='silver/2' (Bank='LP Bank'): 1
- d:\NCKH\Thread_1\data\2024\LP Bank\silver\2\article_texts_20260405_192347_dedup.json


## 5) Tier B — Gọi OpenRouter LLM (OpenAI) để xác nhận ESG + trích context

In [11]:
def load_esg_definition_text(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f"ESG definition not found: {path}")
    txt = path.read_text(encoding="utf-8", errors="ignore").strip()
    if not txt:
        raise ValueError(f"ESG definition file is empty: {path}")
    return txt


ESG_DEFINITION = load_esg_definition_text(ESG_DEFINITION_TXT)
print("Loaded ESG definition chars:", len(ESG_DEFINITION))


Loaded ESG definition chars: 1475


In [12]:
def _get_openrouter_key() -> str:
    key = (OPENROUTER_API_KEY or os.environ.get("OPENROUTER_API_KEY") or "").strip()
    if not key:
        raise RuntimeError(
            "OPENROUTER_API_KEY is empty. Điền vào biến OPENROUTER_API_KEY hoặc set env var OPENROUTER_API_KEY."
        )
    return key


def _extract_json_from_text(text: str) -> dict:
    # Thử parse thẳng; nếu fail thì cắt từ { ... } lớn nhất.
    text = (text or "").strip()
    if not text:
        raise ValueError("Empty response")
    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON object found in response")
    return json.loads(m.group(0))


def call_openrouter_llm(prompt: str, *, session: requests.Session) -> dict:
    key = _get_openrouter_key()
    url = f"{OPENROUTER_BASE_URL}/chat/completions"

    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json",
        # OpenRouter khuyến nghị (không bắt buộc):
        "HTTP-Referer": "http://localhost",
        "X-Title": "NCKH-ESG-Filter",
    }

    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [
            {"role": "system", "content": "Bạn là trợ lý phân loại ESG. Luôn trả về JSON hợp lệ."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.0,
    }

    last_err = None
    for attempt in range(1, LLM_MAX_RETRIES + 1):
        try:
            resp = session.post(url, headers=headers, json=payload, timeout=LLM_TIMEOUT_SECONDS)
            if resp.status_code in {429, 502, 503, 504}:
                raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:200]}")
            resp.raise_for_status()
            data = resp.json()
            content = data["choices"][0]["message"]["content"]
            return _extract_json_from_text(content)
        except Exception as e:
            last_err = e
            if attempt < LLM_MAX_RETRIES:
                time.sleep(2.0 * attempt)
            else:
                raise

    raise RuntimeError(f"LLM call failed: {last_err!r}")


In [13]:
def build_esg_prompt(*, title: str, source: str, link: str, keyword_hits: list[str], context_text: str) -> str:
    import json
    hits = "; ".join(keyword_hits[:30]) if keyword_hits else "(none)"

    schema_example = {
        "is_esg": True,
        "E": False,
        "S": False,
        "G": False,
        "context_esg": "(1–3 câu tiếng Việt; chỉ phần ESG; bám sát context_text)",
        "evidence_quotes": ["(1–2 câu TRÍCH NGUYÊN VĂN từ context_text)", "..."] ,
        "confidence": 0.0,
    }

    return f"""MỤC TIÊU (AI PHÂN LOẠI ESG)
- Một bài được coi là ESG nếu nội dung có nhắc tới ít nhất 1 trong 3 nhóm: E hoặc S hoặc G.
- Không yêu cầu đủ cả 3. Chỉ cần E hoặc S hoặc G là đạt.
- ESG bao gồm cả tác động TÍCH CỰC và TIÊU CỰC (vi phạm, rủi ro, sự cố, tranh chấp…).
- Khi gán nhãn, luôn kèm “bằng chứng” (1–2 câu trích từ bài) nêu rõ lý do.

ĐỊNH NGHĨA ESG (BẮT BUỘC TUÂN THỦ, TRÍCH NGUYÊN VĂN)
{ESG_DEFINITION}

QUY TẮC PHÂN LOẠI:

E — Environmental (Môi trường)
E = tác động tới thiên nhiên/hệ sinh thái, bao gồm cả cải thiện hoặc gây hại.
Các dấu hiệu E:
Phát thải/khí nhà kính/cacbon/Net Zero, biến đổi khí hậu.
Năng lượng tái tạo, tiết kiệm năng lượng, hiệu quả tài nguyên.
Ô nhiễm không khí/nước/đất, sự cố môi trường, xả thải trái phép.
Quản lý chất thải, nhựa, nước thải.
Phá rừng, suy giảm đa dạng sinh học.
Tài chính xanh (ngân hàng): tín dụng xanh, trái phiếu xanh, đánh giá rủi ro môi trường.
Bao gồm cả:
Tích cực: đầu tư năng lượng sạch, giảm phát thải.
Tiêu cực: gây ô nhiễm, vi phạm môi trường, bị xử phạt.


S — Social (Xã hội)
S = cách doanh nghiệp tác động tới con người (nhân viên, khách hàng, cộng đồng), cả tích cực và tiêu cực.
Các dấu hiệu S:
An toàn lao động, tai nạn lao động, điều kiện làm việc.
Đãi ngộ, công bằng, tranh chấp lao động, đình công.
Đa dạng, công bằng, hòa nhập (DEI).
Quyền con người trong chuỗi cung ứng (lao động trẻ em, cưỡng bức…).
Bảo mật dữ liệu, rò rỉ thông tin, quyền riêng tư khách hàng.
Hoạt động cộng đồng/CSR.
Bao gồm cả:
Tích cực: cải thiện phúc lợi, hỗ trợ cộng đồng.
Tiêu cực: vi phạm lao động, rò rỉ dữ liệu, gây hại khách hàng.


G — Governance (Quản trị)
G = hệ thống quản trị, minh bạch, tuân thủ — bao gồm cả vận hành tốt và sai phạm.
Các dấu hiệu G:
Cơ cấu HĐQT, quản trị doanh nghiệp.
Minh bạch tài chính, kiểm toán, công bố thông tin.
Chính sách đạo đức, chống tham nhũng.
Vi phạm pháp luật, gian lận, thao túng, bị điều tra.
Quản trị rủi ro, kiểm soát nội bộ.
Đối với ngân hàng: AML/KYC, tuân thủ, an ninh hệ thống.
Bao gồm cả:
Tích cực: cải thiện minh bạch, nâng cao quản trị.
Tiêu cực: gian lận, tham nhũng, sai phạm quản lý.



YÊU CẦU “CONTEXT ESG” (đoạn trích lý do):
Trả về 1–3 câu ngắn (hoặc đoạn ngắn) nêu rõ nội dung ESG.
Ưu tiên trích nguyên văn (hoặc tóm tắt sát nghĩa)QUY TẮC PHÂN LOẠI:

Nếu bài có nội dung cụ thể liên quan đến môi trường/xã hội/quản trị doanh nghiệp (dù tích cực hay tiêu cực) → gán nhãn tương ứng.
Nếu bài chỉ nói về kết quả kinh doanh, lãi suất, tăng trưởng tín dụng, cổ phiếu, M&A, sản phẩm tài chính… mà KHÔNG có khía cạnh môi trường/xã hội/quản trị → KHÔNG tính ESG.
Khi gán nhãn, luôn kèm “bằng chứng” (1–2 câu trích từ bài) nêu rõ lý do.
E — Environmental (Môi trường)
E = tác động tới thiên nhiên/hệ sinh thái, bao gồm cả cải thiện hoặc gây hại.

Các dấu hiệu E:

Phát thải/khí nhà kính/cacbon/Net Zero, biến đổi khí hậu.
Năng lượng tái tạo, tiết kiệm năng lượng, hiệu quả tài nguyên.
Ô nhiễm không khí/nước/đất, sự cố môi trường, xả thải trái phép.
Quản lý chất thải, nhựa, nước thải.
Phá rừng, suy giảm đa dạng sinh học.
Tài chính xanh (ngân hàng): tín dụng xanh, trái phiếu xanh, đánh giá rủi ro môi trường.

Bao gồm cả:

Tích cực: đầu tư năng lượng sạch, giảm phát thải.
Tiêu cực: gây ô nhiễm, vi phạm môi trường, bị xử phạt.

Không tính E nếu:

Chỉ nhắc “bền vững” chung chung, không có nội dung môi trường cụ thể.
S — Social (Xã hội)
S = cách doanh nghiệp tác động tới con người (nhân viên, khách hàng, cộng đồng), cả tích cực và tiêu cực.

Các dấu hiệu S:

An toàn lao động, tai nạn lao động, điều kiện làm việc.
Đãi ngộ, công bằng, tranh chấp lao động, đình công.
Đa dạng, công bằng, hòa nhập (DEI).
Quyền con người trong chuỗi cung ứng (lao động trẻ em, cưỡng bức…).
Bảo mật dữ liệu, rò rỉ thông tin, quyền riêng tư khách hàng.
Hoạt động cộng đồng/CSR.

Bao gồm cả:

Tích cực: cải thiện phúc lợi, hỗ trợ cộng đồng.
Tiêu cực: vi phạm lao động, rò rỉ dữ liệu, gây hại khách hàng.

Không tính S nếu:

Nội dung quá mơ hồ, không có đối tượng/hành động cụ thể.
G — Governance (Quản trị)
G = hệ thống quản trị, minh bạch, tuân thủ — bao gồm cả vận hành tốt và sai phạm.

Các dấu hiệu G:

Cơ cấu HĐQT, quản trị doanh nghiệp.
Minh bạch tài chính, kiểm toán, công bố thông tin.
Chính sách đạo đức, chống tham nhũng.
Vi phạm pháp luật, gian lận, thao túng, bị điều tra.
Quản trị rủi ro, kiểm soát nội bộ.
Đối với ngân hàng: AML/KYC, tuân thủ, an ninh hệ thống.

Bao gồm cả:

Tích cực: cải thiện minh bạch, nâng cao quản trị.
Tiêu cực: gian lận, tham nhũng, sai phạm quản lý.

Không tính G nếu:

Chỉ nói “quản trị tốt” chung chung, không có chi tiết cụ thể.

YÊU CẦU “CONTEXT ESG” (đoạn trích lý do):

Trả về 1–3 câu ngắn (hoặc đoạn ngắn) nêu rõ nội dung ESG.
Ưu tiên trích nguyên văn (hoặc tóm tắt sát nghĩa).
Nếu không tìm thấy đoạn nào rõ ràng → ESG = False.


THÔNG TIN BÀI
title: {title}
source: {source}
link: {link}
keyword_hits (Tier A): {hits}

context_text (TRÍCH ĐOẠN TỪ BÀI)
{context_text}

JSON schema ví dụ (chỉ tham khảo key)
{json.dumps(schema_example, ensure_ascii=False)}
"""

In [14]:
# Quick preview (no API call)
try:
    ESG_DEFINITION  # noqa: B018
except NameError:
    from pathlib import Path
    _p = Path("Definition_ESG.txt")
    ESG_DEFINITION = _p.read_text(encoding="utf-8") if _p.exists() else "(ESG_DEFINITION not loaded)"

_sample = build_esg_prompt(
    title="BIDV triển khai tín dụng xanh giảm phát thải",
    source="baochinhphu.vn",
    link="https://example.com",
    keyword_hits=["tín dụng xanh", "phát thải"],
    context_text=(
        "Ngân hàng BIDV công bố chương trình tín dụng xanh nhằm giảm phát thải CO2. "
        "Chương trình tập trung vào dự án năng lượng tái tạo và đánh giá rủi ro môi trường trước khi cấp vốn."
    ),
)
print(_sample[:1200])

MỤC TIÊU (AI PHÂN LOẠI ESG)
- Một bài được coi là ESG nếu nội dung có nhắc tới ít nhất 1 trong 3 nhóm: E hoặc S hoặc G.
- Không yêu cầu đủ cả 3. Chỉ cần E hoặc S hoặc G là đạt.
- ESG bao gồm cả tác động TÍCH CỰC và TIÊU CỰC (vi phạm, rủi ro, sự cố, tranh chấp…).
- Khi gán nhãn, luôn kèm “bằng chứng” (1–2 câu trích từ bài) nêu rõ lý do.

ĐỊNH NGHĨA ESG (BẮT BUỘC TUÂN THỦ, TRÍCH NGUYÊN VĂN)
ESG (Environmental, Social, Governance) là một bộ tiêu chí dùng để đánh giá cách một doanh nghiệp hoạt động không chỉ về tài chính mà còn về môi trường, xã hội và quản trị nội bộ.

1. Environmental (E) – Môi trường

Đánh giá tác động của doanh nghiệp đến môi trường tự nhiên.

Bao gồm: phát thải khí nhà kính, ô nhiễm, sử dụng tài nguyên, năng lượng tái tạo, đa dạng sinh học.
Tích cực: giảm phát thải, sử dụng tài nguyên hiệu quả, bảo vệ môi trường.
Tiêu cực: gây ô nhiễm, khai thác tài nguyên quá mức, góp phần vào biến đổi khí hậu.
2. Social (S) – Xã hội

Đánh giá cách doanh nghiệp đối xử với con người v

In [15]:
# Guard: dừng sớm nếu chưa có API key (tránh crawl lâu rồi mới fail ở Tier B)
_key = (OPENROUTER_API_KEY or os.environ.get("OPENROUTER_API_KEY") or "").strip()
if not _key:
    raise RuntimeError(
        "OPENROUTER_API_KEY đang trống. Hãy điền vào biến OPENROUTER_API_KEY (Cell cấu hình) "
        "hoặc set env var OPENROUTER_API_KEY, rồi chạy lại từ cell này trở xuống."
    )

## 6) Chạy pipeline: Tier A → Tier B → lưu JSON vào `silver/3/`

In [16]:
def _to_bool(v) -> bool:
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return v != 0
    if isinstance(v, str):
        s = v.strip().casefold()
        if s in {"true", "yes", "y", "1"}:
            return True
        if s in {"false", "no", "n", "0", ""}:
            return False
    return False


def group_inputs_by_bank(input_paths: list[Path]) -> dict[Path, list[Path]]:
    out: dict[Path, list[Path]] = {}
    for p in input_paths:
        bank_dir = _infer_bank_dir_from_path(p)
        out.setdefault(bank_dir, []).append(p)
    for k in out:
        out[k] = sorted(out[k])
    return out


def load_articles_for_bank(paths: list[Path]) -> list[InputArticle]:
    merged: list[InputArticle] = []
    for p in paths:
        if p.suffix.lower() == ".json" or str(STAGE_IN).casefold().startswith("silver"):
            merged.extend(load_articles_from_silver_json(p))
        else:
            merged.extend(load_articles_from_bronze_excel(p))
    # dedupe by link across files
    seen = set()
    dedup: list[InputArticle] = []
    for a in merged:
        if a.link in seen:
            continue
        seen.add(a.link)
        dedup.append(a)
    if MAX_ARTICLES_PER_BANK is not None:
        dedup = dedup[: int(MAX_ARTICLES_PER_BANK)]
    return dedup


def run_pipeline_for_bank(bank_dir: Path, input_paths_for_bank: list[Path]) -> list[dict]:
    articles = load_articles_for_bank(input_paths_for_bank)
    if not articles:
        return []

    # HTTP sessions (crawl chỉ dùng khi input không có content sẵn)
    crawl_sess = requests.Session()
    crawl_sess.headers.update(
        {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            )
        }
    )
    llm_sess = requests.Session()

    out_items: list[dict] = []
    total = len(articles)
    tier_a_pass = 0
    llm_kept = 0

    for idx, a in enumerate(articles, start=1):
        print(f"[{a.bank}] ({idx}/{total}) {a.title}")
        print(f"  link: {a.link}")

        content = (a.content or "").strip()
        if not content:
            # Nếu input là bronze/excel thì cần crawl; nếu input là silver mà thiếu content thì bỏ qua
            content, crawl_method = fetch_article_text(a.link, session=crawl_sess, timeout=HTTP_TIMEOUT_SECONDS)
            time.sleep(max(0.0, float(CRAWL_SLEEP_SECONDS)))
            if not content:
                print("  -> content missing (skip)")
                continue

        # Tier A: keyword hits
        tier_a_text = f"{a.title}\n{content}"
        hits = tier_a_find_hits(tier_a_text)
        if not hits:
            print("  -> Tier A: no keyword hits (skip)")
            continue
        tier_a_pass += 1

        # Build context text for LLM
        context_text = extract_context_sentences(content, hits)
        if not context_text:
            context_text = content[:MAX_CONTENT_CHARS_FOR_LLM]
        else:
            head = content[:1500]
            combo = (head + "\n\n" + context_text).strip()
            context_text = combo[:MAX_CONTENT_CHARS_FOR_LLM]

        prompt = build_esg_prompt(
            title=a.title,
            source=a.source,
            link=a.link,
            keyword_hits=hits,
            context_text=context_text,
        )

        # Tier B: LLM confirm
        try:
            out = call_openrouter_llm(prompt, session=llm_sess)
        except Exception as e:
            print("  -> Tier B LLM ERROR:", repr(e))
            continue
        finally:
            time.sleep(max(0.0, float(LLM_SLEEP_SECONDS)))

        is_esg = _to_bool(out.get("is_esg"))
        context_esg = str(out.get("context_esg") or "").strip()
        if not is_esg or not context_esg:
            print("  -> Tier B: not ESG / empty context (skip)")
            continue

        E = _to_bool(out.get("E"))
        S = _to_bool(out.get("S"))
        G = _to_bool(out.get("G"))
        if not (E or S or G):
            print("  -> Tier B: E/S/G all false (skip)")
            continue

        evidence_quotes = out.get("evidence_quotes") or []
        if not isinstance(evidence_quotes, list):
            evidence_quotes = [str(evidence_quotes)]

        item = {
            "link": a.link,
            "source": a.source,
            "title": a.title,
            "content": content,
            "E": E,
            "S": S,
            "G": G,
            "context_esg": context_esg,
            "evidence_quotes": evidence_quotes[:3],
            "confidence": out.get("confidence"),
        }
        out_items.append(item)
        llm_kept += 1
        print("  -> KEEP (ESG)")

    print(f"\n[{bank_dir.name}] Summary: total={total}, tierA_pass={tier_a_pass}, kept={llm_kept}")
    return out_items


def save_stage3_json(bank_dir: Path, items: list[dict]) -> Path:
    out_dir = bank_dir / STAGE_OUT
    out_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = out_dir / f"esg_check_llms_{ts}.json"
    payload = {
        "bank": bank_dir.name,
        "year": bank_dir.parent.name,
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "model": OPENROUTER_MODEL,
        "dictionary": str(ESG_DICT_XLSX),
        "definition": str(ESG_DEFINITION_TXT),
        "stage_in": STAGE_IN,
        "stage_out": STAGE_OUT,
        "items": items,
    }
    out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return out_path


if not input_paths:
    print(f"No input files found for STAGE_IN='{STAGE_IN}'.")
    if str(STAGE_IN).casefold() == "bronze":
        print("Expected: data/<YEAR>/<BANK>/bronze/*_filter.xlsx")
    else:
        print("Expected: data/<YEAR>/<BANK>/silver/2/*.json")
else:
    saved_paths: list[Path] = []
    by_bank = group_inputs_by_bank(input_paths)
    for bank_dir, paths in by_bank.items():
        print("\n=== BANK:", bank_dir.name, "| YEAR:", bank_dir.parent.name, "===")
        print("Inputs:")
        for p in paths:
            print("-", p)
        items = run_pipeline_for_bank(bank_dir, paths)
        out_path = save_stage3_json(bank_dir, items)
        print("Saved stage3 JSON:", out_path)
        saved_paths.append(out_path)

    print("\nDone. Saved files:")
    for p in saved_paths:
        print("-", p)


=== BANK: LP Bank | YEAR: 2024 ===
Inputs:
- d:\NCKH\Thread_1\data\2024\LP Bank\silver\2\article_texts_20260405_192347_dedup.json
[LP Bank] (1/61) LPBank bổ nhiệm cố vấn cấp cao ban điều hành - VnExpress
  link: https://vnexpress.net/lpbank-bo-nhiem-co-van-cap-cao-ban-dieu-hanh-4777807.html
  -> KEEP (ESG)
[LP Bank] (2/61) LPBank chỉ có hai cổ đông nắm trên 1% cổ phần nhà băng - VnExpress
  link: https://vnexpress.net/lpbank-chi-co-hai-co-dong-nam-tren-1-co-phan-nha-bang-4774133.html
  -> KEEP (ESG)
[LP Bank] (3/61) LPBank đổi tên thành Ngân hàng Lộc Phát Việt Nam - VnExpress
  link: https://vnexpress.net/lpbank-doi-ten-thanh-ngan-hang-loc-phat-viet-nam-4770617.html
  -> KEEP (ESG)
[LP Bank] (4/61) LPBank công bố Hoàng Đức làm đại sứ thương hiệu - VnExpress
  link: https://vnexpress.net/lpbank-cong-bo-hoang-duc-lam-dai-su-thuong-hieu-4771286.html
  -> KEEP (ESG)
[LP Bank] (5/61) LPBank tăng vốn điều lệ, bổ sung hội đồng quản trị - VnExpress
  link: https://vnexpress.net/lpbank-tang-vo